# Advanced Python: solution / reference

Working code for the advanced recovery-prediction exercises.

## Setup

Run this cell first.

In [ ]:
import os, subprocess, sys

REPO = "mjmarte/ai-coding-workshop"
if os.path.isdir(".git"):
    result = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif os.path.isdir("ai-coding-workshop"):
    os.chdir("ai-coding-workshop")
    result = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif not os.path.exists("data"):
    result = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"])
    if result.returncode == 0:
        os.chdir("ai-coding-workshop")
else:
    result = subprocess.run(["git", "status"], capture_output=True)
if result.returncode != 0:
    raise SystemExit("Could not download or update the workshop files. Check the GitHub link.")

missing = [path for path in ("data/recovery_prediction.csv",) if not os.path.exists(path)]
if missing:
    raise SystemExit(f"Setup incomplete. Missing: {missing}")
print("Ready.")

---
## Task A1 - Define the prediction question before fitting a model

Goal: identify the outcome, specify predictors available at the acute assessment,
and confirm one row per participant.

The prediction question requires a time boundary. The 12-month outcome and later measurements
are excluded from the acute predictor matrix.

In [ ]:
import pandas as pd

recovery = pd.read_csv("data/recovery_prediction.csv")
outcome = "outcome_wab_aq_12m"
clinical_features = [
    "age", "education_years", "acute_wab_aq", "acute_discourse_score",
]
multimodal_features = clinical_features + [
    "lesion_volume_ml", "cortical_lesion_pct",
    "dorsal_disconnection_pct", "ventral_disconnection_pct",
]

assert recovery["participant_id"].is_unique
print("participants:", len(recovery))
print(recovery.isna().sum())

> Expected: 90 participants, one row per participant, and no outcome or identifier in either predictor list.

---
## Task A2 - Compare baseline and multimodal models without leakage

Goal: compare a clinical baseline model with a model that additionally includes acute
lesion and disconnection measures.

Imputation, scaling, and fitting occur within each resampling fold so that held-out records do
not influence the training procedure.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=202607)

def evaluate_feature_set(features):
    model = make_pipeline(
        SimpleImputer(),
        StandardScaler(),
        RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
    )
    scores = cross_validate(
        model,
        recovery[features],
        recovery[outcome],
        cv=cv,
        scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    )
    return {
        "MAE_mean": -scores["test_mae"].mean(),
        "MAE_sd": scores["test_mae"].std(),
        "R2_mean": scores["test_r2"].mean(),
        "R2_sd": scores["test_r2"].std(),
    }

cv_summary = pd.DataFrame(
    [evaluate_feature_set(clinical_features), evaluate_feature_set(multimodal_features)],
    index=["clinical", "clinical_plus_imaging"],
).round(2)
print(cv_summary)

> Expected: clinical MAE 5.25 (SD 0.71), R-squared 0.81 (SD 0.08); clinical-plus-imaging MAE 4.85 (SD 0.84), R-squared 0.82 (SD 0.08). These are synthetic development results, not evidence of clinical utility.

---
## Task A3 - Inspect out-of-fold predictions and error distribution

Goal: generate held-out predictions and inspect their errors alongside the resampled summary.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_predict

oof_model = make_pipeline(
    SimpleImputer(),
    StandardScaler(),
    RidgeCV(alphas=[0.01, 0.1, 1, 10, 100]),
)
oof_cv = KFold(n_splits=5, shuffle=True, random_state=202607)
oof_pred = cross_val_predict(
    oof_model, recovery[multimodal_features], recovery[outcome], cv=oof_cv,
)

audit = recovery[["sex", outcome]].copy()
audit["predicted_wab_aq_12m"] = oof_pred
audit["absolute_error"] = (audit[outcome] - audit["predicted_wab_aq_12m"]).abs()

print("overall MAE:", round(mean_absolute_error(audit[outcome], oof_pred), 2))
print(audit.groupby("sex")["absolute_error"].agg(["count", "mean"]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(audit[outcome], audit["predicted_wab_aq_12m"], alpha=0.7)
axes[0].plot([20, 100], [20, 100], color="black", linewidth=1)
axes[0].set(xlabel="Observed 12-month WAB-AQ", ylabel="Out-of-fold prediction")

for x, group in enumerate(["F", "M"]):
    values = audit.loc[audit["sex"] == group, "absolute_error"]
    axes[1].scatter([x] * len(values), values, alpha=0.7)
axes[1].set(xticks=[0, 1], xticklabels=["F", "M"], ylabel="Absolute prediction error")
fig.tight_layout()
plt.show()

> Expected: overall out-of-fold MAE 4.84. Each point in the first panel was predicted without fitting that participant. The displayed subgroup errors are descriptive.

---
## Task A4 - Fact-check an AI-written prediction summary

Goal: constrain an AI-written summary to the resampling output and the scope of a
synthetic development exercise.

Cross-validation estimates performance under the specified resampling procedure. It does not
establish clinical utility, causal contribution, external validity, calibration, or deployment readiness.

In [ ]:
# Do not run code for this task. Paste the prompt into the AI assistant, then compare every
# number and every claim in its response against cv_summary and the stated scope of the dataset.

> A correct response reports only the resampled metrics shown in `cv_summary` and makes no clinical, causal, or external-validation claim.

---
## Task A5 - Use an agent for a bounded reproducibility audit

Goal: ask an agent to inspect the project without authority to alter data or
analysis files.

An agent can inventory files and identify consistency questions. It does not determine the
predictor set or the validity of a clinical claim.

In [ ]:
# Do not run code for this task. Paste the prompt into a coding agent with read-only
# repository access, then inspect the cited file paths yourself.

> A useful agent response cites the generator, the CSV, this notebook, and the resampling code. It does not make clinical claims or edit the project.